# Real-Time Financial Anomaly Detection

This notebook mirrors the runnable project. It uses fraud labels only for evaluation, not for fitting unsupervised models.

## Introduction, Objectives, and Dataset

The Credit Card Fraud dataset has `Time`, PCA-transformed `V1-V28`, `Amount`, and `Class`. RobustScaler is applied to amount because it is resistant to extreme values. PCA below is visualization-only; the V columns are not transformed again for modeling.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.data_loader import load_creditcard_data
from src.preprocessing import prepare_experiment
from src.threshold_tuning import tune_thresholds
from src.isolation_forest_model import train_isolation_forest, score_isolation_forest, predict_isolation_forest
from src.lof_model import train_lof, score_lof, predict_lof
from src.evaluation import evaluate_predictions


## Load, understand, clean, and preprocess

Place the CSV in `data/creditcard.csv` before running. The loader reports a clear error if it is missing or malformed.

In [2]:
data = load_creditcard_data()
print(data.shape)
display(data.head())
display(data.dtypes)
print('Missing:', data.isna().sum().sum(), 'Duplicates:', data.duplicated().sum())
display(data['Class'].value_counts())
experiment = prepare_experiment(data)
print('Baseline rows:', len(experiment.baseline_features), 'Validation rows:', len(experiment.validation_features), 'Test rows:', len(experiment.test_features))


(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Missing: 0 Duplicates: 1081


Class
0    284315
1       492
Name: count, dtype: int64

Baseline rows: 169951 Validation rows: 56745 Test rows: 56746


## Isolation Forest and contamination tuning

Contamination is the expected outlier fraction. We calculate metrics for several candidates rather than asserting a value is best.

In [3]:
isolation_forest, if_time = train_isolation_forest(experiment.baseline_features)
if_baseline_scores = score_isolation_forest(isolation_forest, experiment.baseline_features)
if_validation_scores = score_isolation_forest(isolation_forest, experiment.validation_features)
tuning = tune_thresholds(if_baseline_scores, if_validation_scores, experiment.validation_labels, 'Isolation Forest')
display(tuning)
if_threshold = float(tuning.iloc[0]['Score Threshold'])
if_pred, if_scores = predict_isolation_forest(isolation_forest, experiment.test_features, if_threshold)


,Model,Precision,Recall,F1 Score,False Positive Rate,False Negative Rate,TP,TN,FP,FN,Execution Time (seconds),Average Precision,Contamination,Score Threshold
0,Isolation Forest,0.104046,0.382979,0.163636,0.005472,0.617021,36,56341,310,58,0.0,0.081478,0.005,0.602156
1,Isolation Forest,0.129771,0.180851,0.151111,0.002012,0.819149,17,56537,114,77,0.0,0.081478,0.002,0.638365
2,Isolation Forest,0.083207,0.585106,0.145695,0.010697,0.414894,55,56045,606,39,0.0,0.081478,0.010,0.573779
3,Isolation Forest,0.051906,0.680851,0.096458,0.020635,0.319149,64,55482,1169,30,0.0,0.081478,0.020,0.536744
4,Isolation Forest,0.116667,0.074468,0.090909,0.000936,0.925532,7,56598,53,87,0.0,0.081478,0.001,0.664819
5,Isolation Forest,0.040861,0.787234,0.077690,0.030661,0.212766,74,54914,1737,20,0.0,0.081478,0.030,0.514244
6,Isolation Forest,0.026207,0.808511,0.050768,0.049849,0.191489,76,53827,2824,18,0.0,0.081478,0.050,0.486934


## LOF, evaluation, results, limitations, and future scope

LOF compares a transaction's local density to neighboring points. Precision limits false alarms, recall measures fraud found, and F1 balances them. Accuracy is avoided as the primary metric because normal transactions dominate.

In [4]:
lof, lof_time = train_lof(experiment.baseline_features, 'auto')
lof_baseline_scores = score_lof(lof, experiment.baseline_features)
lof_validation_scores = score_lof(lof, experiment.validation_features)
lof_tuning = tune_thresholds(lof_baseline_scores, lof_validation_scores, experiment.validation_labels, 'LOF')
lof_threshold = float(lof_tuning.iloc[0]['Score Threshold'])
lof_pred, lof_scores = predict_lof(lof, experiment.test_features, lof_threshold)
import pandas as pd
comparison = pd.DataFrame([
    evaluate_predictions(experiment.test_labels, if_pred, if_scores, 'Isolation Forest', if_time),
    evaluate_predictions(experiment.test_labels, lof_pred, lof_scores, 'LOF', lof_time),
])
display(comparison)
# For complete EDA, plots, saved models, PR curves and PCA visualization, run: python run.py


,Model,Precision,Recall,F1 Score,False Positive Rate,False Negative Rate,TP,TN,FP,FN,Execution Time (seconds),Average Precision
0,Isolation Forest,0.123494,0.431579,0.192037,0.005137,0.568421,41,56360,291,54,0.37895,0.112150
1,LOF,0.521008,0.652632,0.579439,0.001006,0.347368,62,56594,57,33,1.10160,0.524513
